# FADB-CGF: Frequency-Aware Dual-Branch Network with Confidence-Gated Fusion
### Brain Tumor MRI Classification — Kaggle Experiment Notebook

This notebook trains and rigorously compares:
1. Baseline EfficientNetB0 (fine-tuned)
2. Baseline ResNet50 (fine-tuned)
3. Frequency-branch-only ablation
4. FADB-CGF with concatenation fusion
5. FADB-CGF with confidence-gated cross-attention fusion

Each model is trained across 5 seeds. See the accompanying `FADB-CGF_Kaggle_Guideline.md`
for the full experiment protocol, session planning, and honest-reporting checklist.

**Run cells top to bottom.** Training cells check for existing checkpoints and resume
automatically if a session is interrupted — see Section 6 (Training Loop).

## 1. Setup & Configuration

In [ ]:
!pip install imagehash --quiet


In [ ]:
import os, json, random, time, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# ---- Paths: auto-detect the mounted dataset folder instead of hardcoding it ----
# Kaggle mounts each added dataset under /kaggle/input/<slug>/, and the slug can
# differ slightly from what you'd guess (versioning, renames, etc). We search for
# it by looking for Training/Testing subfolders rather than assuming a fixed name.

INPUT_ROOT = Path("/kaggle/input")
print("Available datasets under /kaggle/input:")
for p in INPUT_ROOT.iterdir():
    print(" -", p.name)

def find_data_root(input_root):
    candidates = []
    for p in input_root.rglob("*"):
        if p.is_dir() and p.name.lower() in ("training", "train"):
            sibling_test = None
            for sib in p.parent.iterdir():
                if sib.is_dir() and sib.name.lower() in ("testing", "test"):
                    sibling_test = sib
                    break
            if sibling_test is not None:
                candidates.append((p.parent, p, sibling_test))
    return candidates

candidates = find_data_root(INPUT_ROOT)
if not candidates:
    raise FileNotFoundError(
        "Could not auto-locate a Training/Testing folder pair under /kaggle/input.\n"
        "Fix: check the printed dataset list above, open it in the Kaggle file browser "
        "(right sidebar), and manually set DATA_ROOT below to the correct path, e.g.\n"
        "  DATA_ROOT = Path('/kaggle/input/<exact-folder-name>')"
    )

DATA_ROOT, TRAIN_DIR, TEST_DIR = candidates[0]
print(f"\nAuto-detected dataset root: {DATA_ROOT}")
print(f"Train dir: {TRAIN_DIR}")
print(f"Test dir:  {TEST_DIR}")

if len(candidates) > 1:
    print("\nWARNING: multiple Training/Testing pairs found — verify this picked the right one:")
    for c in candidates:
        print("  ", c[0])

WORK_DIR = Path("/kaggle/working")
CKPT_DIR = WORK_DIR / "checkpoints"
RESULTS_DIR = WORK_DIR / "results"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "confusion_matrices").mkdir(exist_ok=True)

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15          # reduced from 20 - baseline converges by ~epoch 9, so 15 gives margin without wasting quota
LR = 1e-4
WEIGHT_DECAY = 1e-4
SEEDS = [42, 123, 456, 789]   # reduced from 5 to 4 seeds

# Fail fast and loud if the class subfolders don't actually exist under TRAIN_DIR/TEST_DIR
actual_train_classes = {p.name.lower() for p in TRAIN_DIR.iterdir() if p.is_dir()} if TRAIN_DIR.exists() else set()
missing = [c for c in CLASSES if c not in actual_train_classes]
if missing:
    raise FileNotFoundError(
        f"TRAIN_DIR exists but is missing expected class folders: {missing}\n"
        f"Found instead: {sorted(actual_train_classes)}\n"
        f"Fix: update the CLASSES list above to match the actual folder names, "
        f"or check that {TRAIN_DIR} is really the Training directory."
    )
print("\nClass folders verified OK.")


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


## 2. Exploratory Checks (run once, before any training)

Verify class counts, then run a perceptual-hash duplicate check across the
train/test split. This dataset is a merge of three source datasets — leakage
across splits is a realistic risk and must be checked, not assumed absent.

In [ ]:
# ---- Class count check ----
def count_images(root):
    counts = {}
    for c in CLASSES:
        p = root / c
        counts[c] = len(list(p.glob("*.jpg"))) + len(list(p.glob("*.jpeg"))) + len(list(p.glob("*.png")))
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)
print("Train counts:", train_counts, "Total:", sum(train_counts.values()))
print("Test counts:", test_counts, "Total:", sum(test_counts.values()))
grand_total = sum(train_counts.values()) + sum(test_counts.values())
print("Grand total:", grand_total, "(expected 7023)")

if grand_total == 0:
    raise RuntimeError(
        "Zero images found across all class folders. Do not proceed - every downstream "
        "cell will silently run on empty data. Re-check TRAIN_DIR/TEST_DIR and CLASSES "
        "against the actual folder structure printed above."
    )


In [ ]:
# ---- Near-duplicate / leakage check via perceptual hashing (Hamming-distance based) ----
# Exact-hash matching (distance=0) misses near-duplicates created by augmentation
# (slight rotation/crop/brightness shift of the same source slice). We compute
# pairwise Hamming distance between all test and train pHashes and flag anything
# within a small threshold. Results are cached to disk so re-running the notebook
# across sessions doesn't require re-hashing every time.

import imagehash

HAMMING_THRESHOLD = 5   # out of 64 bits; <=5 is a conservative near-duplicate cutoff
LEAK_CACHE_PATH = RESULTS_DIR / "leakage_check.json"

def hash_folder(root):
    records = []  # list of (path_str, class_name, hash_hex)
    for c in CLASSES:
        for f in sorted((root / c).glob("*")):
            if f.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                continue
            try:
                h = imagehash.phash(Image.open(f).convert("L"))
                records.append((str(f), c, str(h)))
            except Exception as e:
                print("skip", f, e)
    return records


POPCOUNT_TABLE = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint16)

def hex_hashes_to_uint8_matrix(hex_list):
    ints = np.array([int(h, 16) for h in hex_list], dtype=np.uint64)
    return ints.view(np.uint8).reshape(-1, 8)

def pairwise_hamming(train_bytes, test_bytes):
    """Returns (n_test, n_train) Hamming distance matrix."""
    xor = train_bytes[None, :, :] ^ test_bytes[:, None, :]   # (n_test, n_train, 8)
    dist = POPCOUNT_TABLE[xor].sum(axis=2)                    # (n_test, n_train)
    return dist


if LEAK_CACHE_PATH.exists():
    print(f"Loading cached leakage check from {LEAK_CACHE_PATH}")
    with open(LEAK_CACHE_PATH) as f:
        cache = json.load(f)
    leaks = cache["leaks"]                 # list of [test_path, class, matched_train_path, distance]
    n_test_hashed = cache["n_test_hashed"]
    within_train_dupes = cache.get("within_train_dupe_count", None)
else:
    print("Hashing training set...")
    train_records = hash_folder(TRAIN_DIR)
    print("Hashing test set...")
    test_records = hash_folder(TEST_DIR)

    train_bytes = hex_hashes_to_uint8_matrix([r[2] for r in train_records])
    test_bytes = hex_hashes_to_uint8_matrix([r[2] for r in test_records])

    print(f"Computing pairwise Hamming distances ({len(test_records)} x {len(train_records)})...")
    dist_matrix = pairwise_hamming(train_bytes, test_bytes)   # (n_test, n_train)
    min_dist = dist_matrix.min(axis=1)
    argmin_train = dist_matrix.argmin(axis=1)

    leaks = []
    for i, (test_path, test_class, test_hash) in enumerate(test_records):
        if min_dist[i] <= HAMMING_THRESHOLD:
            train_path, train_class, train_hash = train_records[argmin_train[i]]
            leaks.append([test_path, test_class, train_path, int(min_dist[i])])

    # Optional: within-train near-duplicate count (informational only — not auto-removed,
    # since redundant augmented copies in train don't bias test accuracy, but are worth knowing)
    print("Checking within-train near-duplicate rate (sampled)...")
    sample_n = min(1500, len(train_records))
    rng = np.random.default_rng(0)
    sample_idx = rng.choice(len(train_records), size=sample_n, replace=False)
    sample_bytes = train_bytes[sample_idx]
    sample_dist = pairwise_hamming(train_bytes, sample_bytes)
    # Note: fancy indexing (sample_dist[:, sample_idx]) returns a copy in numpy, so
    # np.fill_diagonal on that slice would silently no-op. Zero out self-matches directly instead.
    sample_dist[np.arange(sample_n), sample_idx] = 255
    within_train_dupes = int((sample_dist.min(axis=1) <= HAMMING_THRESHOLD).sum())
    n_test_hashed = len(test_records)

    with open(LEAK_CACHE_PATH, "w") as f:
        json.dump({
            "leaks": leaks,
            "n_test_hashed": n_test_hashed,
            "hamming_threshold": HAMMING_THRESHOLD,
            "within_train_dupe_count": within_train_dupes,
            "within_train_sample_size": sample_n,
        }, f, indent=2)
    print(f"Leakage check cached to {LEAK_CACHE_PATH}")

print(f"\nPotential train/test near-duplicates found (Hamming <= {HAMMING_THRESHOLD}): {len(leaks)} / {n_test_hashed} test images ({100*len(leaks)/max(n_test_hashed,1):.1f}%)")
if within_train_dupes is not None:
    print(f"Within-train near-duplicate rate (sampled): {within_train_dupes} flagged in a {cache['within_train_sample_size'] if 'cache' in dir() else sample_n}-image sample — informational, not auto-removed.")

if leaks:
    print("\nFirst few flagged pairs (test_path, class, matched_train_path, hamming_distance):")
    for row in leaks[:10]:
        print(" ", row)
    per_class_leak = defaultdict(int)
    for _, c, _, _ in leaks:
        per_class_leak[c] += 1
    print("\nLeaks per class:", dict(per_class_leak))
else:
    print("No near-duplicates found within the Hamming threshold.")


### 2.1 Second-stage verification: pixel-level SSIM (reduces false positives)

pHash within a small Hamming distance means "structurally similar," not necessarily
"the same image." Low-texture classes (e.g. `notumor` — largely uniform, healthy
brain scans) are prone to pHash **false-positive collisions**: genuinely different
scans that hash close together simply because they lack distinguishing detail.

Before trusting a pHash match as real leakage, verify it at the pixel level with
SSIM (structural similarity). True near-duplicates from augmentation (rotate/crop/
brightness shift of the *same* source slice) will have very high SSIM even after
those transforms. Two distinct patients' scans that merely look similar will not.

Only pairs passing **both** checks (Hamming distance AND high SSIM) are treated as
confirmed leakage. This is a stricter, more defensible remedy than pHash alone.

In [ ]:
!pip install scikit-image --quiet


In [ ]:
from skimage.metrics import structural_similarity as ssim

SSIM_THRESHOLD = 0.90   # pairs below this are treated as false-positive pHash collisions, not real leaks
SSIM_CACHE_PATH = RESULTS_DIR / "leakage_ssim_verified.json"

def load_gray_array(path, size=128):
    img = Image.open(path).convert("L").resize((size, size))
    return np.array(img, dtype=np.float64) / 255.0

if SSIM_CACHE_PATH.exists():
    print(f"Loading cached SSIM-verified leaks from {SSIM_CACHE_PATH}")
    with open(SSIM_CACHE_PATH) as f:
        verified_leaks = json.load(f)["verified_leaks"]
else:
    print(f"Running SSIM verification on {len(leaks)} candidate pairs...")
    verified_leaks = []
    rejected = []
    for test_path, cls, train_path, dist in leaks:
        a = load_gray_array(test_path)
        b = load_gray_array(train_path)
        score = ssim(a, b, data_range=1.0)
        if score >= SSIM_THRESHOLD:
            verified_leaks.append([test_path, cls, train_path, dist, float(score)])
        else:
            rejected.append([test_path, cls, train_path, dist, float(score)])

    with open(SSIM_CACHE_PATH, "w") as f:
        json.dump({
            "verified_leaks": verified_leaks,
            "rejected_false_positives": rejected,
            "ssim_threshold": SSIM_THRESHOLD,
        }, f, indent=2)
    print(f"SSIM verification cached to {SSIM_CACHE_PATH}")

print(f"\npHash candidates: {len(leaks)}")
print(f"Confirmed leaks after SSIM verification (>= {SSIM_THRESHOLD}): {len(verified_leaks)}")
print(f"Rejected as pHash false positives: {len(leaks) - len(verified_leaks)}")

verified_per_class = defaultdict(int)
for _, c, _, _, _ in verified_leaks:
    verified_per_class[c] += 1
print("\nConfirmed leaks per class:", dict(verified_per_class))


### 2.2 Remedy: build a leakage-filtered test set using SSIM-confirmed leaks

The remedy applied here is exclusion — but only for pairs confirmed as genuine
near-duplicates by both pHash proximity AND pixel-level SSIM, not raw pHash alone.
The confirmed exclusion list is persisted so it's identical and reproducible across
every training run and every seed — this must be the same filtered test set for
every model compared, or the comparison isn't fair.

In [ ]:
leak_paths = {row[0] for row in verified_leaks}

# Verify class balance survives filtering (a skewed leak rate across classes could
# otherwise bias per-class metrics even after filtering)
post_filter_counts = defaultdict(int)
for c in CLASSES:
    total_c = test_counts[c]
    leaked_c = sum(1 for row in verified_leaks if row[1] == c)
    post_filter_counts[c] = total_c - leaked_c

print("Test set counts BEFORE filtering:", dict(test_counts))
print("Test set counts AFTER filtering: ", dict(post_filter_counts))
print("Total test images after filtering:", sum(post_filter_counts.values()), "of", sum(test_counts.values()))

max_c, min_c = max(post_filter_counts.values()), min(post_filter_counts.values())
if max_c > 0 and (max_c - min_c) / max_c > 0.15:
    print("\nWARNING: leakage removal is unevenly distributed across classes (>15% relative spread).")
    print("Per-class metrics on the filtered test set may be less comparable across classes. Report this explicitly.")
else:
    print("\nClass balance after filtering looks reasonably even — safe to proceed.")


### 2.3 Sanity check: visually inspect a few confirmed pairs

Confirm the SSIM-verified pairs are genuinely near-duplicates before trusting the
filtered test set.

In [ ]:
import matplotlib.pyplot as plt

n_show = min(4, len(verified_leaks))
if n_show > 0:
    fig, axes = plt.subplots(n_show, 2, figsize=(6, 3 * n_show))
    if n_show == 1:
        axes = axes.reshape(1, 2)
    for i in range(n_show):
        test_path, cls, train_path, dist, score = verified_leaks[i]
        axes[i, 0].imshow(Image.open(test_path).convert("L"), cmap="gray")
        axes[i, 0].set_title(f"TEST: {Path(test_path).name}\n({cls})")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(Image.open(train_path).convert("L"), cmap="gray")
        axes[i, 1].set_title(f"TRAIN match: {Path(train_path).name}\nHamming={dist}, SSIM={score:.3f}")
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "leakage_examples.png", dpi=120)
    plt.show()
else:
    print("No confirmed leaks to display.")


## 3. Dataset & Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def compute_fft_magnitude(pil_img, size=IMG_SIZE):
    """Log-magnitude 2D FFT spectrum, returned as a single-channel tensor."""
    img = pil_img.convert("L").resize((size, size))
    arr = np.array(img, dtype=np.float32)
    f = np.fft.fft2(arr)
    fshift = np.fft.fftshift(f)
    magnitude = np.log1p(np.abs(fshift))
    magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-8)
    return torch.from_numpy(magnitude).unsqueeze(0).float()  # (1, H, W)


class BrainTumorDataset(Dataset):
    """Returns (spatial_tensor, freq_tensor, label). Excludes any leaked files if provided."""
    def __init__(self, root, transform, exclude_files=None):
        self.samples = []
        exclude_files = set(exclude_files or [])
        for c in CLASSES:
            for f in sorted((root / c).glob("*")):
                if f.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                    continue
                if str(f) in exclude_files:
                    continue
                self.samples.append((f, CLASS_TO_IDX[c]))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        spatial = self.transform(img)
        freq = compute_fft_magnitude(img)
        return spatial, freq, label


train_dataset = BrainTumorDataset(TRAIN_DIR, train_transform)
test_dataset = BrainTumorDataset(TEST_DIR, eval_transform, exclude_files=leak_paths)

print("Train size:", len(train_dataset), "Test size (post leakage filter):", len(test_dataset))
assert len(test_dataset) == sum(post_filter_counts.values()), "Filtered test set size mismatch — investigate before training."


## 4. Model Definitions

In [ ]:
class SpatialBranch(nn.Module):
    """ImageNet-pretrained backbone, returns pooled feature embedding."""
    def __init__(self, backbone="efficientnet_b0"):
        super().__init__()
        if backbone == "efficientnet_b0":
            net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            self.features = net.features
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.out_dim = 1280
        elif backbone == "resnet50":
            net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
            self.features = nn.Sequential(*list(net.children())[:-2])
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.out_dim = 2048
        else:
            raise ValueError(backbone)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return x


class FrequencyBranch(nn.Module):
    """Lightweight CNN over the FFT-magnitude spectrum (1-channel input)."""
    def __init__(self, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, out_dim, 3, padding=1), nn.BatchNorm2d(out_dim), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.out_dim = out_dim

    def forward(self, x):
        return self.net(x).flatten(1)


class ConcatFusionHead(nn.Module):
    def __init__(self, spatial_dim, freq_dim, num_classes=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(spatial_dim + freq_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, spatial_feat, freq_feat):
        return self.fc(torch.cat([spatial_feat, freq_feat], dim=1))


class GatedFusionHead(nn.Module):
    """Confidence-gated cross-attention: a learned per-sample scalar gate weights
    the frequency embedding's contribution relative to the spatial embedding."""
    def __init__(self, spatial_dim, freq_dim, num_classes=4, proj_dim=256):
        super().__init__()
        self.spatial_proj = nn.Linear(spatial_dim, proj_dim)
        self.freq_proj = nn.Linear(freq_dim, proj_dim)
        self.gate = nn.Sequential(
            nn.Linear(proj_dim * 2, proj_dim), nn.ReLU(), nn.Linear(proj_dim, 1), nn.Sigmoid()
        )
        self.fc = nn.Sequential(
            nn.Linear(proj_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, spatial_feat, freq_feat):
        s = self.spatial_proj(spatial_feat)
        f = self.freq_proj(freq_feat)
        g = self.gate(torch.cat([s, f], dim=1))  # (B, 1) in [0, 1]
        fused = s + g * f
        return self.fc(fused)


class FADB_CGF(nn.Module):
    def __init__(self, backbone="efficientnet_b0", fusion="gated", num_classes=4):
        super().__init__()
        self.spatial_branch = SpatialBranch(backbone)
        self.freq_branch = FrequencyBranch()
        if fusion == "concat":
            self.head = ConcatFusionHead(self.spatial_branch.out_dim, self.freq_branch.out_dim, num_classes)
        elif fusion == "gated":
            self.head = GatedFusionHead(self.spatial_branch.out_dim, self.freq_branch.out_dim, num_classes)
        else:
            raise ValueError(fusion)

    def forward(self, spatial_x, freq_x):
        s = self.spatial_branch(spatial_x)
        f = self.freq_branch(freq_x)
        return self.head(s, f)


class FrequencyOnlyModel(nn.Module):
    """Ablation: does the frequency branch carry signal on its own?"""
    def __init__(self, num_classes=4):
        super().__init__()
        self.freq_branch = FrequencyBranch()
        self.fc = nn.Sequential(nn.Linear(self.freq_branch.out_dim, 128), nn.ReLU(), nn.Linear(128, num_classes))

    def forward(self, spatial_x, freq_x):
        return self.fc(self.freq_branch(freq_x))


class BaselineModel(nn.Module):
    """Spatial-only baseline (EfficientNetB0 or ResNet50)."""
    def __init__(self, backbone="efficientnet_b0", num_classes=4):
        super().__init__()
        self.spatial_branch = SpatialBranch(backbone)
        self.fc = nn.Linear(self.spatial_branch.out_dim, num_classes)

    def forward(self, spatial_x, freq_x=None):
        return self.fc(self.spatial_branch(spatial_x))


## 5. Train / Test Split Setup (from Training folder, held-out val)

In [ ]:
from torch.utils.data import random_split

def make_loaders(seed, val_frac=0.15):
    set_seed(seed)
    n_val = int(len(train_dataset) * val_frac)
    n_train = len(train_dataset) - n_val
    train_sub, val_sub = random_split(
        train_dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(seed)
    )
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader


## 6. Training Loop (with checkpoint resumption)

Checkpoints are saved every epoch. If a Kaggle session is interrupted, re-running
this cell picks up from the last saved epoch for that (model_name, seed) pair.

In [ ]:
def train_one_run(model_fn, model_name, seed, epochs=EPOCHS, lr=LR):
    ckpt_path = CKPT_DIR / f"{model_name}_seed{seed}.pt"
    set_seed(seed)
    train_loader, val_loader, test_loader = make_loaders(seed)

    model = model_fn().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    start_epoch = 0
    best_val_acc = 0.0
    history = []

    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optim_state"])
        scheduler.load_state_dict(ckpt["sched_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt.get("best_val_acc", 0.0)
        history = ckpt.get("history", [])
        print(f"[{model_name} seed{seed}] Resuming from epoch {start_epoch}")
        if start_epoch >= epochs:
            print(f"[{model_name} seed{seed}] Already complete.")
            return model, history

    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for spatial_x, freq_x, labels in train_loader:
            spatial_x, freq_x, labels = spatial_x.to(device), freq_x.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                outputs = model(spatial_x, freq_x)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()
        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for spatial_x, freq_x, labels in val_loader:
                spatial_x, freq_x, labels = spatial_x.to(device), freq_x.to(device), labels.to(device)
                outputs = model(spatial_x, freq_x)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total
        best_val_acc = max(best_val_acc, val_acc)
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_acc": val_acc})
        print(f"[{model_name} seed{seed}] epoch {epoch+1}/{epochs} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "sched_state": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
            "history": history,
        }, ckpt_path)

    return model, history


## 7. Evaluation Utilities

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for spatial_x, freq_x, labels in test_loader:
            spatial_x, freq_x = spatial_x.to(device), freq_x.to(device)
            outputs = model(spatial_x, freq_x)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    prec, rec, f1, support = precision_recall_fscore_support(all_labels, all_preds, labels=range(len(CLASSES)))
    cm = confusion_matrix(all_labels, all_preds, labels=range(len(CLASSES)))

    gm_confusion = cm[CLASS_TO_IDX["glioma"], CLASS_TO_IDX["meningioma"]] + cm[CLASS_TO_IDX["meningioma"], CLASS_TO_IDX["glioma"]]

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "per_class_precision": prec.tolist(),
        "per_class_recall": rec.tolist(),
        "per_class_f1": f1.tolist(),
        "confusion_matrix": cm.tolist(),
        "glioma_meningioma_confusion": int(gm_confusion),
        "predictions": all_preds.tolist(),
        "labels": all_labels.tolist(),
    }


## 8. Run the Full Multi-Seed Sweep

**Session planning note:** this cell runs everything sequentially. Given Kaggle's
12-hour session limit and 30 GPU-hr/week quota, you will likely want to comment out
sections and run them across separate sessions per the guideline's session plan.
Checkpointing means re-running is safe and will skip completed runs.

In [ ]:
MODEL_REGISTRY = {
    "efficientnet_baseline": lambda: BaselineModel("efficientnet_b0"),
    "resnet50_baseline": lambda: BaselineModel("resnet50"),
    "frequency_only": lambda: FrequencyOnlyModel(),
    "fadb_cgf_concat": lambda: FADB_CGF("efficientnet_b0", fusion="concat"),
    "fadb_cgf_gated": lambda: FADB_CGF("efficientnet_b0", fusion="gated"),
}

all_results = defaultdict(list)  # model_name -> list of per-seed result dicts
results_csv_path = RESULTS_DIR / "per_run_metrics.csv"

# Resume from partial results if this cell was already run in a prior session
if results_csv_path.exists():
    prior_df = pd.read_csv(results_csv_path)
    print(f"Found {len(prior_df)} prior results, will skip completed (model, seed) pairs.")
else:
    prior_df = pd.DataFrame(columns=["model_name", "seed", "accuracy", "macro_f1", "gm_confusion"])

completed = set(zip(prior_df["model_name"], prior_df["seed"])) if len(prior_df) else set()

rows = prior_df.to_dict("records")

for model_name, model_fn in MODEL_REGISTRY.items():
    for seed in SEEDS:
        if (model_name, seed) in completed:
            print(f"Skipping {model_name} seed {seed} (already completed)")
            continue
        print(f"\n=== Training {model_name} | seed {seed} ===")
        model, history = train_one_run(model_fn, model_name, seed)
        _, _, test_loader = make_loaders(seed)
        result = evaluate_model(model, test_loader)
        all_results[model_name].append(result)

        rows.append({
            "model_name": model_name, "seed": seed,
            "accuracy": result["accuracy"], "macro_f1": result["macro_f1"],
            "gm_confusion": result["glioma_meningioma_confusion"],
        })
        pd.DataFrame(rows).to_csv(results_csv_path, index=False)

        cm_path = RESULTS_DIR / "confusion_matrices" / f"{model_name}_seed{seed}.npy"
        np.save(cm_path, np.array(result["confusion_matrix"]))

        with open(RESULTS_DIR / f"{model_name}_seed{seed}_full.json", "w") as f:
            json.dump(result, f)

        del model
        torch.cuda.empty_cache()

print("\nSweep complete (or resumed to completion). See", results_csv_path)


## 9. Aggregate Results Table (mean ± std across seeds)

In [ ]:
df = pd.read_csv(results_csv_path)
summary = df.groupby("model_name").agg(
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    gm_confusion_mean=("gm_confusion", "mean"), gm_confusion_std=("gm_confusion", "std"),
).reset_index()
summary.to_csv(RESULTS_DIR / "summary_table.csv", index=False)
summary


## 10. Statistical Significance Testing

McNemar's test compares paired predictions (same test set) between two models for
a single seed. Wilcoxon signed-rank compares accuracy across the 5 seeds. Report
both — accuracy differences that aren't significant should be reported as such.

In [ ]:
def mcnemar_test(preds_a, preds_b, labels):
    a_correct = (np.array(preds_a) == np.array(labels))
    b_correct = (np.array(preds_b) == np.array(labels))
    both_correct = np.sum(a_correct & b_correct)
    a_only = np.sum(a_correct & ~b_correct)
    b_only = np.sum(~a_correct & b_correct)
    neither = np.sum(~a_correct & ~b_correct)
    table = [[both_correct, a_only], [b_only, neither]]
    result = mcnemar(table, exact=(a_only + b_only) < 25)
    return {"statistic": float(result.statistic), "p_value": float(result.pvalue), "table": table}


def load_full_result(model_name, seed):
    with open(RESULTS_DIR / f"{model_name}_seed{seed}_full.json") as f:
        return json.load(f)


stat_results = {}
comparison_pairs = [
    ("fadb_cgf_gated", "efficientnet_baseline"),
    ("fadb_cgf_gated", "resnet50_baseline"),
    ("fadb_cgf_gated", "fadb_cgf_concat"),
    ("fadb_cgf_concat", "efficientnet_baseline"),
]

for model_a, model_b in comparison_pairs:
    key = f"{model_a}_vs_{model_b}"
    stat_results[key] = {}

    # McNemar's per seed (using the same seed's test predictions for both models)
    mcnemar_per_seed = []
    for seed in SEEDS:
        try:
            ra = load_full_result(model_a, seed)
            rb = load_full_result(model_b, seed)
            mc = mcnemar_test(ra["predictions"], rb["predictions"], ra["labels"])
            mcnemar_per_seed.append(mc)
        except FileNotFoundError:
            continue
    stat_results[key]["mcnemar_per_seed"] = mcnemar_per_seed

    # Wilcoxon across seeds' accuracy
    acc_a = df[df.model_name == model_a].sort_values("seed")["accuracy"].values
    acc_b = df[df.model_name == model_b].sort_values("seed")["accuracy"].values
    if len(acc_a) == len(acc_b) and len(acc_a) > 1:
        try:
            w_stat, w_p = wilcoxon(acc_a, acc_b)
            stat_results[key]["wilcoxon"] = {"statistic": float(w_stat), "p_value": float(w_p)}
        except ValueError as e:
            stat_results[key]["wilcoxon"] = {"error": str(e)}

with open(RESULTS_DIR / "statistical_tests.json", "w") as f:
    json.dump(stat_results, f, indent=2)

for k, v in stat_results.items():
    print(k)
    if "wilcoxon" in v:
        print("  Wilcoxon:", v["wilcoxon"])
    if v.get("mcnemar_per_seed"):
        avg_p = np.mean([m["p_value"] for m in v["mcnemar_per_seed"]])
        print(f"  McNemar mean p-value across seeds: {avg_p:.4f}")


## 11. Robustness Evaluation

Clean-test accuracy on this benchmark is likely saturated across all models. This is
where a genuine difference between architectures is more likely to show up — evaluate
every trained model under noise, blur, and brightness perturbations.

In [ ]:
# Preflight check: confirm the objects this cell needs are actually in memory.
# If any are missing, it means Sections 1-8 haven't been (fully) re-run in this
# kernel session - re-run those (definitions only, fast, no retraining) then retry.
_required = ["BaselineModel", "FrequencyOnlyModel", "FADB_CGF", "CKPT_DIR", "RESULTS_DIR",
             "SEEDS", "device", "make_loaders", "accuracy_score", "torch", "pd"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise NameError(
        f"Missing required objects: {_missing}. This means Sections 1-8 (setup through "
        f"evaluation utilities) haven't all been executed in this kernel session yet. "
        f"Re-run those cells (they only define things and load checkpoint paths - they "
        f"will NOT retrain anything), then re-run this cell. You do NOT need to re-run "
        f"Section 9 (the training sweep) - that would needlessly repeat the full sweep."
    )
print("Preflight check passed - all required objects present.")


In [ ]:
# NOTE: This cell is self-contained and does not depend on Section 9 (the training
# sweep) having been run in this kernel session. If you restarted the kernel after
# training completed, you only need Sections 1-8 (setup, dataset, model definitions,
# loaders, training/eval function definitions) re-run before this cell - NOT Section 9,
# since that would needlessly re-trigger the full sweep loop. MODEL_REGISTRY is
# redefined here for that reason.
MODEL_REGISTRY = {
    "efficientnet_baseline": lambda: BaselineModel("efficientnet_b0"),
    "resnet50_baseline": lambda: BaselineModel("resnet50"),
    "frequency_only": lambda: FrequencyOnlyModel(),
    "fadb_cgf_concat": lambda: FADB_CGF("efficientnet_b0", fusion="concat"),
    "fadb_cgf_gated": lambda: FADB_CGF("efficientnet_b0", fusion="gated"),
}

import torchvision.transforms.functional as TF

def add_gaussian_noise(img_tensor, std=0.05):
    return img_tensor + torch.randn_like(img_tensor) * std

def apply_perturbation(spatial_x, kind):
    if kind == "noise":
        return add_gaussian_noise(spatial_x, std=0.08)
    elif kind == "blur":
        return TF.gaussian_blur(spatial_x, kernel_size=[5, 5], sigma=[2.0, 2.0])
    elif kind == "brightness":
        return torch.clamp(spatial_x * 1.3, spatial_x.min(), spatial_x.max())
    else:
        return spatial_x


def evaluate_robustness(model, test_loader, kinds=("clean", "noise", "blur", "brightness")):
    model.eval()
    results = {}
    for kind in kinds:
        all_preds, all_labels = [], []
        with torch.no_grad():
            for spatial_x, freq_x, labels in test_loader:
                spatial_x, freq_x = spatial_x.to(device), freq_x.to(device)
                if kind != "clean":
                    spatial_x = apply_perturbation(spatial_x, kind)
                outputs = model(spatial_x, freq_x)
                all_preds.extend(outputs.argmax(1).cpu().numpy())
                all_labels.extend(labels.numpy())
        results[kind] = accuracy_score(all_labels, all_preds)
    return results


robustness_rows = []
for model_name, model_fn in MODEL_REGISTRY.items():
    for seed in SEEDS:
        ckpt_path = CKPT_DIR / f"{model_name}_seed{seed}.pt"
        if not ckpt_path.exists():
            continue
        model = model_fn().to(device)
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        _, _, test_loader = make_loaders(seed)
        rob = evaluate_robustness(model, test_loader)
        rob["model_name"] = model_name
        rob["seed"] = seed
        robustness_rows.append(rob)
        del model
        torch.cuda.empty_cache()
        print(model_name, seed, rob)

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(RESULTS_DIR / "robustness_results.csv", index=False)
robustness_df.groupby("model_name")[["clean", "noise", "blur", "brightness"]].mean()


## 12. Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_REGISTRY), figsize=(24, 5))
for ax, model_name in zip(axes, MODEL_REGISTRY.keys()):
    cms = []
    for seed in SEEDS:
        p = RESULTS_DIR / "confusion_matrices" / f"{model_name}_seed{seed}.npy"
        if p.exists():
            cms.append(np.load(p))
    if not cms:
        continue
    mean_cm = np.mean(cms, axis=0)
    sns.heatmap(mean_cm, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, cbar=False)
    ax.set_title(model_name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices_summary.png", dpi=150)
plt.show()


## 13. Honest-Reporting Summary

Run this cell last. It prints a plain-language summary comparing FADB-CGF to the
best baseline — use this as the seed of your results section, adjusted for whatever
the numbers actually show.

In [ ]:
best_baseline = summary.loc[summary[summary.model_name.isin(["efficientnet_baseline", "resnet50_baseline"])]["accuracy_mean"].idxmax()]
gated_row = summary[summary.model_name == "fadb_cgf_gated"].iloc[0]
concat_row = summary[summary.model_name == "fadb_cgf_concat"].iloc[0]
freq_only_row = summary[summary.model_name == "frequency_only"].iloc[0]

print(f"Best baseline: {best_baseline['model_name']} — accuracy {best_baseline['accuracy_mean']:.4f} ± {best_baseline['accuracy_std']:.4f}")
print(f"FADB-CGF (gated): accuracy {gated_row['accuracy_mean']:.4f} ± {gated_row['accuracy_std']:.4f}")
print(f"FADB-CGF (concat): accuracy {concat_row['accuracy_mean']:.4f} ± {concat_row['accuracy_std']:.4f}")
print(f"Frequency-only ablation: accuracy {freq_only_row['accuracy_mean']:.4f} ± {freq_only_row['accuracy_std']:.4f}")

diff = gated_row['accuracy_mean'] - best_baseline['accuracy_mean']
print(f"\nGated fusion vs best baseline: {'+' if diff >= 0 else ''}{diff:.4f} accuracy")
print("Check statistical_tests.json for whether this difference is significant before claiming improvement.")
print("Check robustness_results.csv for whether differences emerge under perturbation even if clean accuracy is tied.")
print("Check gm_confusion columns for whether the glioma/meningioma confusion specifically improved.")
